# hl7pet_arrow: Apache Arrow / PyArrow / PySpark demo

Demonstrates spec `6002-arrow-integration`'s two extraction mechanisms --
single-PATH (`extract_value`) and multi-PATH (`extract_values`) -- against
a standalone PyArrow Table and a local PySpark DataFrame, for both
non-hierarchy and hierarchy-mode (`->`) PATHs, and finishes with a
comparison against the existing row-by-row plain `hl7pet` binding.

## Setup

In [ ]:
import json
from pathlib import Path

import pyarrow as pa

import hl7pet
import hl7pet_arrow as ha
import hl7pet_arrow.spark as has

FIXTURES = Path("..") / "fixtures"

## Load HL7 messages into a standalone PyArrow Table

One message per row, plus a `None` row to show null-row handling
(a missing message never raises -- it's just another `"no_match"` row).

In [ ]:
messages = [
    (FIXTURES / "messages/baseline.hl7").read_text(),
    (FIXTURES / "messages/filter-example.hl7").read_text(),
    None,
]
table = pa.table({"message": pa.array(messages)})
table

## Single-PATH extraction: `extract_value`

One PATH, applied to the whole column at once -- one Arrow `StructArray`
back, `{value, status}` per row.

In [ ]:
result = ha.extract_value(table["message"].combine_chunks(), "MSH-12")
result.to_pylist()

## Multi-PATH extraction: `extract_values`

Several PATHs, computed from a single scan per message -- one outer
struct field per requested PATH.

In [ ]:
paths = ["MSH-12", "PID-5.1", "OBX-5"]
multi = ha.extract_values(table["message"].combine_chunks(), paths)
for i, path in enumerate(paths):
    print(path, "->", multi.field(i).to_pylist())

## Hierarchy-mode PATHs via both mechanisms

A `->` PATH requires a `segmentDefinition` profile, the same shape the
existing plain `hl7pet.get_value_hierarchy` already consumes.

In [ ]:
hierarchy_message = (FIXTURES / "messages/basic-hierarchy.hl7").read_text()
profile = json.loads((FIXTURES / "profiles/basic-two-level.json").read_text())
hier_table = pa.table({"message": pa.array([hierarchy_message])})

single_hier = ha.extract_value(
    hier_table["message"].combine_chunks(), "OBR[1] -> OBX-5", profile
)
print("extract_value (hierarchy):", single_hier.to_pylist())

multi_hier = ha.extract_values(
    hier_table["message"].combine_chunks(),
    ["OBR[1] -> OBX-5", "OBR[2] -> OBX-5"],
    profile,
)
for i, path in enumerate(["OBR[1] -> OBX-5", "OBR[2] -> OBX-5"]):
    print(path, "->", multi_hier.field(i).to_pylist())

## The same mechanisms from a local PySpark session

`hl7pet_arrow.spark.extract_value_udf`/`extract_values_udf` wrap the
above as ordinary PySpark column functions (`arrow_udf`-based, no pandas
conversion), usable with `.withColumn`/`.select` like any other column.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("hl7pet_arrow-demo")
    .master("local[1]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

df = spark.createDataFrame([(m,) for m in messages], ["message"])
df.withColumn("MSH_12", has.extract_value_udf("MSH-12")(df["message"])).show(
    truncate=False
)

In [ ]:
multi_df = df.select(
    has.extract_values_udf(["MSH-12", "PID-5.1"])(df["message"]).alias("result")
)
multi_df.show(truncate=False)
multi_df.schema

## Comparison: columnar `hl7pet_arrow` vs. row-by-row plain `hl7pet`

Confirms both mechanisms reproduce exactly what the existing
`hl7pet.get_value`/`get_values` return, one message at a time -- this is
spec `6002`'s FR-004 ("changes how extraction is invoked, not what it
returns") demonstrated directly, not just asserted.

In [ ]:
path = "MSH-12"
row_by_row = [hl7pet.get_value(m, path) if m is not None else None for m in messages]
columnar = [
    (row["value"] if row["status"] == "ok" else None)
    for row in ha.extract_value(pa.array(messages), path).to_pylist()
]
print("row-by-row (hl7pet.get_value):", row_by_row)
print("columnar   (hl7pet_arrow.extract_value):", columnar)
assert row_by_row == columnar
print("MATCH")

In [ ]:
spark.stop()